# Zoe.Logos-Graph — Schema and Annotation Guidelines

This notebook documents the extraction schema and annotation principles for v1.

Use this as the reference document for all annotators and for evaluating LLM extraction quality.

---

## Core principle

> **Extract only what is supported by the abstract. Do not infer. Do not invent.**

The scientific value of Zoe.Logos-Graph comes from disciplined structure, not from vague generation.

## Field-by-field guidelines

### `paper_id`
- Assign a unique identifier: `paper_001`, `paper_002`, etc., or a DOI slug.
- Must be unique across the corpus.

### `title`
- Copy the full title exactly as given.

### `year`
- Use the publication year. Set to `null` if not available.

### `species_common_name` / `species_scientific_name` / `taxonomic_family`
- Use the most specific species mentioned as the focal species.
- If the study is comparative across multiple species, use `'multiple species'` for common name and `'multiple'` for scientific name.
- Use the full binomial (`Genus species`) or trinomial (`Genus species subspecies`) name.
- Use `'unknown'` only when the abstract genuinely does not name a species.

### `developmental_stage`

| Value | When to use |
|---|---|
| `embryo` | Pre-hatching / pre-birth |
| `early-life` | Hatchling, nestling, pup, neonatal |
| `juvenile` | Post-early-life, pre-adult |
| `adult` | Sexually mature animals |
| `mixed` | Study explicitly includes multiple stages |
| `unknown` | Stage not stated |

### `communication_domain`
- `vocal`: study primarily concerns vocal signals
- `multimodal`: study explicitly combines vocal and non-vocal signals
- `unknown`: not clear from abstract

### `vocalisation_type`
- List all types of vocalisation described.
- Normalise to lowercase consistent terms: `'call'`, `'alarm call'`, `'song'`, `'subsong'`, `'whistle'`, `'ultrasonic vocalisation'`, etc.
- Do not include 'vocalisation' alone if a more specific type is named.
- Leave empty `[]` only if the abstract does not describe any specific vocalisation type.

### `behavioural_context`
- List contexts in which vocalisation occurs or is elicited.
- Examples: `'foraging'`, `'predator response'`, `'courtship'`, `'parent-offspring interaction'`, `'isolation'`, `'group cohesion'`
- Use consistent lowercase labels. Refer to the normalisation map in `src/normalisation.py`.

### `putative_function`
- List communicative functions attributed to the vocalisation **in the abstract**.
- Only include functions that the paper itself proposes, not general background knowledge.
- Examples: `'mate attraction'`, `'individual recognition'`, `'predator warning'`, `'maternal retrieval'`

### `analysis_method`
- List all analytical or computational methods mentioned.
- Include both signal processing (e.g. `'spectrogram analysis'`) and statistical/ML methods (e.g. `'UMAP'`, `'k-means clustering'`, `'ANOVA'`).
- Do not include general terms like `'statistics'` unless no specific method is named.

### `main_outcome`
- 1–2 sentences summarising the main finding.
- Faithful to the abstract. Do not add interpretation.
- Do not start with 'This paper...' or 'The study...'.
- Keep under 200 characters if possible.

### `dataset_or_recording_available`
- `yes`: the abstract states that data or recordings are deposited or available.
- `no`: the abstract explicitly states that data are not available.
- `unknown`: no statement about data availability.

### `dataset_name`
- If a named repository or dataset is mentioned (e.g. `'xeno-canto'`, `'Dryad'`, `'OSF'`), record it here.
- Set to `null` otherwise.

### `notes_uncertainty`
- Use for any ambiguity, inference, or caveat in the extraction.
- Set to `null` when the extraction is unambiguous.


## Common mistakes to avoid

| Mistake | Correct approach |
|---|---|
| Inferring species from context not in abstract | Use `'unknown'` |
| Adding functions the paper does not claim | Only extract functions stated in the abstract |
| Using singular/plural inconsistently | Normalise: always `'call'`, not `'calls'` |
| Copying the abstract as `main_outcome` | Write a concise faithful summary |
| Setting `dataset_or_recording_available: 'yes'` without a named source | Only mark `yes` if stated in the abstract |

## Validation

Run the validator on your annotation file:

```bash
python -m src.validation --input data/annotations/pilot.json
```

Fix all errors before proceeding. Review all warnings.

In [ ]:
# Load and display the pilot dataset
import json
import pandas as pd

with open('../data/annotations/pilot.json') as f:
    pilot = json.load(f)

df = pd.DataFrame(pilot)
df[['paper_id', 'title', 'species_common_name', 'developmental_stage', 'vocalisation_type', 'main_outcome']]

In [ ]:
# Run validation programmatically
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.validation import validate_file, soft_checks, print_report

valid, errors = validate_file(Path('../data/annotations/pilot.json'))
warnings = {r.paper_id: w for r in valid if (w := soft_checks(r))}
print_report(valid, errors, warnings)